# 01 — Verify graph walk against cdisc-org/COSMoS LinkML YAML

Reconnaissance notebook for the `cosmos-graph-walk` branch — Step 2 prep.

**Purpose.** Walk the authored LinkML YAML of the 2026-Q1 package programmatically, to see the actual graph shape before designing the graph-walk flattener. Nothing in this notebook is a pipeline output; nothing here changes `interim/COSMoS_BC_DSS.xlsx`. Pure reconnaissance.

**Scope.**

1. Introspect `cosmos_sdtm_model.yaml` via `SchemaView` — classes, slots, enums.
2. Walk the cached sample `sdtm_abi.yaml` and validate it against the schema.
3. Download all `yaml/20260331_r16/sdtm/*.yaml` (227 files) into `downloads/cosmos_yaml/20260331_r16/sdtm/`.
4. Distribution of the `source` slot across 227 DSSs — how many distinct VLM pinning patterns exist.
5. Per-DSS reification coverage — which DSSs carry `assignedTerm`, `codelist`, `subsetCodelist`, `valueList` at variable level, and how often.
6. One walked example per distinct `source` pattern.
7. Validate all 227 DSSs against the schema.

**Scope exclusions.** BC side (bc/*.yaml), DEC resolution, cross-domain traversal, flattener output, xlsx shape decisions. Those are follow-up notebooks.

**References.**

- [cosmos-bc-dss/docs/COSMoS_Graph_As_Authored.md](../docs/COSMoS_Graph_As_Authored.md) — Step 1, §9 verification checklist.
- [cosmos-bc-dss/docs/COSMoS_Flattener_Enhancement_Brief.md](../docs/COSMoS_Flattener_Enhancement_Brief.md) — motivating brief.
- Cached schemas at `reference/cosmos_linkml/` (gitignored). Live source is `github.com/cdisc-org/COSMoS`.


## 1. Imports


In [1]:
import sys
from pathlib import Path
from collections import Counter

import requests
import yaml
import pandas as pd

from linkml_runtime.utils.schemaview import SchemaView
from linkml.validator import validate

print('python  :', sys.version.split()[0])
print('pandas  :', pd.__version__)


python  : 3.12.6
pandas  : 3.0.1


## 2. Configuration

Schema lives in the repo-level `reference/cosmos_linkml/` cache (gitignored). Downloaded YAML goes to `cosmos-bc-dss/downloads/cosmos_yaml/<package>/sdtm/` (also gitignored per CLAUDE.md).


In [2]:
REPO_ROOT = Path.cwd().parent.parent  # notebooks/ -> cosmos-bc-dss/ -> repo root
SCHEMA_DIR = REPO_ROOT / 'reference' / 'cosmos_linkml'
SDTM_SCHEMA = SCHEMA_DIR / 'cosmos_sdtm_model.yaml'
SAMPLE_ABI = SCHEMA_DIR / 'sdtm_abi.yaml'

PACKAGE_DATE = '20260331_r16'
GH_API = 'https://api.github.com/repos/cdisc-org/COSMoS/contents/yaml'
GH_RAW = 'https://raw.githubusercontent.com/cdisc-org/COSMoS/main/yaml'

CACHE_DIR = Path('..') / 'downloads' / 'cosmos_yaml' / PACKAGE_DATE / 'sdtm'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

assert SDTM_SCHEMA.exists(), f'schema not found: {SDTM_SCHEMA}'
assert SAMPLE_ABI.exists(), f'sample not found: {SAMPLE_ABI}'
print('schema :', SDTM_SCHEMA)
print('sample :', SAMPLE_ABI)
print('cache  :', CACHE_DIR.resolve())


schema : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/reference/cosmos_linkml/cosmos_sdtm_model.yaml
sample : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/reference/cosmos_linkml/sdtm_abi.yaml
cache  : /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/cosmos-bc-dss/downloads/cosmos_yaml/20260331_r16/sdtm


## 3. Introspect the SDTM schema

Classes, slots, enumerations. The slot lists on `SDTMGroup` and `SDTMVariable` are what the graph-walk flattener will enumerate for column generation in Step 2.


In [3]:
sv = SchemaView(str(SDTM_SCHEMA))

classes = list(sv.all_classes())
slots = list(sv.all_slots())
enums = list(sv.all_enums())
tree_root = [c for c in classes if sv.get_class(c).tree_root]

print(f'classes   : {len(classes):3d}  {classes}')
print(f'slots     : {len(slots):3d}')
print(f'enums     : {len(enums):3d}  {enums}')
print(f'tree_root : {tree_root}')


classes   :   7  ['SDTMGroup', 'SDTMVariable', 'RelationShip', 'CodeList', 'CodeListTerm', 'SubsetCodeList', 'AssignedTerm']
slots     :  43
enums     :   8  ['PackageTypeEnum', 'SDTMVariableDataTypeEnum', 'LinkingPhraseEnum', 'PredicateTermEnum', 'OriginTypeEnum', 'OriginSourceEnum', 'RoleEnum', 'ComparatorEnum']
tree_root : ['SDTMGroup']


In [4]:
print('SDTMGroup induced slots:')
for s in sv.class_induced_slots('SDTMGroup'):
    print(f'  {s.name:32s}  range={s.range}')

print()
print('SDTMVariable induced slots:')
for s in sv.class_induced_slots('SDTMVariable'):
    print(f'  {s.name:32s}  range={s.range}')


SDTMGroup induced slots:
  packageDate                       range=date
  packageType                       range=PackageTypeEnum
  datasetSpecializationId           range=string
  domain                            range=string
  shortName                         range=string
  source                            range=string
  sdtmigStartVersion                range=string
  sdtmigEndVersion                  range=string
  biomedicalConceptId               range=string
  variables                         range=SDTMVariable

SDTMVariable induced slots:
  name                              range=string
  dataElementConceptId              range=string
  isNonStandard                     range=boolean
  codelist                          range=CodeList
  subsetCodelist                    range=string
  valueList                         range=string
  assignedTerm                      range=AssignedTerm
  role                              range=RoleEnum
  relationship                      rang

In [5]:
# The NCIt-anchored enumerations the flattener will surface directly
for enum_name in ('OriginTypeEnum', 'OriginSourceEnum', 'RoleEnum', 'ComparatorEnum'):
    e = sv.get_enum(enum_name)
    pvs = list(e.permissible_values.values())
    print(f'{enum_name}  ({len(pvs)} values)')
    for pv in pvs:
        meaning = pv.meaning or ''
        print(f'  {pv.text:20s}  meaning={meaning}')
    print()


OriginTypeEnum  (5 values)
  Assigned              meaning=NCIT:C170547
  Collected             meaning=NCIT:C170548
  Derived               meaning=NCIT:C170549
  Predecessor           meaning=NCIT:C170550
  Protocol              meaning=NCIT:C170551

OriginSourceEnum  (4 values)
  Investigator          meaning=NCIT:C25936
  Sponsor               meaning=NCIT:C70793
  Subject               meaning=NCIT:C41189
  Vendor                meaning=NCIT:C68608

RoleEnum  (4 values)
  Identifier            meaning=
  Qualifier             meaning=
  Timing                meaning=
  Topic                 meaning=

ComparatorEnum  (2 values)
  EQ                    meaning=
  IN                    meaning=



## 4. Walk one real DSS — sdtm_abi.yaml

First confirmation that the cached sample parses, carries the slots the schema declares, and validates clean.


In [6]:
abi = yaml.safe_load(SAMPLE_ABI.read_text())
print('top-level keys :', sorted(abi.keys()))
print('DS_Code        :', abi['datasetSpecializationId'])
print('source         :', abi['source'])
print('BC ID          :', abi['biomedicalConceptId'])
print('n variables    :', len(abi['variables']))


top-level keys : ['biomedicalConceptId', 'datasetSpecializationId', 'domain', 'packageDate', 'packageType', 'sdtmigEndVersion', 'sdtmigStartVersion', 'shortName', 'source', 'variables']
DS_Code        : ABI
source         : VS.VSTESTCD
BC ID          : C87304
n variables    : 6


In [7]:
# Variable-level reification — which slots are populated on each variable?
for v in abi['variables']:
    reified = [k for k in ('assignedTerm','codelist','subsetCodelist','valueList','relationship') if v.get(k)]
    at = v.get('assignedTerm') or {}
    print(f"  {v['name']:14s} role={v.get('role','-'):10s} "
          f"mVar={str(v.get('mandatoryVariable','-')):5s} mVal={str(v.get('mandatoryValue','-')):5s} "
          f"cmp={str(v.get('comparator','-')):4s} pinned={str(at.get('value','')):10s} "
          f"reified={','.join(reified)}")


  VSTESTCD       role=Topic      mVar=True  mVal=True  cmp=EQ   pinned=ABI        reified=assignedTerm,codelist,relationship
  VSTEST         role=Qualifier  mVar=True  mVal=True  cmp=-    pinned=Ankle-Brachial Index reified=assignedTerm,codelist,relationship
  VSORRES        role=Qualifier  mVar=True  mVal=False cmp=-    pinned=           reified=relationship
  VSSTRESC       role=Qualifier  mVar=False mVal=False cmp=-    pinned=           reified=relationship
  VSSTRESN       role=Qualifier  mVar=False mVal=False cmp=-    pinned=           reified=relationship
  VSDTC          role=Timing     mVar=True  mVal=False cmp=-    pinned=           reified=relationship


In [8]:
# Validate ABI against the schema
report = validate(abi, str(SDTM_SCHEMA), target_class='SDTMGroup')
print(f'ABI validation: {len(report.results)} issue(s)')
for r in report.results:
    print(f'  {r.severity}: {r.message}')


ABI validation: 0 issue(s)


## 5. Download all sdtm/*.yaml for the 2026-Q1 package

227 files according to the §9-v1 verification in the Step 1 doc. Cached on first run; no-op thereafter. Re-downloadable at any time (gitignored).


In [9]:
listing_url = f'{GH_API}/{PACKAGE_DATE}/sdtm'
resp = requests.get(listing_url, timeout=15)
resp.raise_for_status()
entries = resp.json()
yaml_files = sorted(e['name'] for e in entries if e['type'] == 'file' and e['name'].endswith('.yaml'))
print(f'yaml/{PACKAGE_DATE}/sdtm/ : {len(yaml_files)} yaml files')
print('first 5:', yaml_files[:5])


yaml/20260331_r16/sdtm/ : 227 yaml files
first 5: ['sdtm_abi.yaml', 'sdtm_absknf.yaml', 'sdtm_addrelf_re.yaml', 'sdtm_advevent.yaml', 'sdtm_airtrap_re.yaml']


In [10]:
session = requests.Session()
downloaded = 0
for name in yaml_files:
    target = CACHE_DIR / name
    if target.exists():
        continue
    r = session.get(f'{GH_RAW}/{PACKAGE_DATE}/sdtm/{name}', timeout=15)
    r.raise_for_status()
    target.write_bytes(r.content)
    downloaded += 1

cached = sorted(CACHE_DIR.glob('*.yaml'))
print(f'downloaded this run: {downloaded}')
print(f'total cached       : {len(cached)}')


downloaded this run: 227
total cached       : 227


## 6. Source-pattern distribution

The `source` slot — the `<domain>.<variable>` a DSS pins on — is the hinge between the DSS and the SDTM IG. How many distinct patterns exist across 227 DSSs?


In [11]:
all_dss = {}
for p in sorted(CACHE_DIR.glob('*.yaml')):
    d = yaml.safe_load(p.read_text())
    all_dss[d['datasetSpecializationId']] = d

print(f'parsed DSSs: {len(all_dss)}')

source_counts = Counter(d.get('source') for d in all_dss.values())
no_source = [k for k, d in all_dss.items() if not d.get('source')]
print(f'distinct source values: {len(source_counts)}')
print(f'DSSs without source   : {len(no_source)}')
print()
print(f"{'source':32s} count")
for src, n in source_counts.most_common():
    print(f'  {str(src):30s} {n:4d}')


parsed DSSs: 227
distinct source values: 5
DSSs without source   : 0

source                           count
  RE.RETESTCD                     135
  VS.VSTESTCD                      63
  DS.DECODE                        14
  LB.LBTESTCD                      13
  GF.GFTESTCD                       2


## 7. Reification coverage inventory

One row per DSS, counting how many variables carry each of the reified child objects. Shows where the flattener's projection currently drops information — any non-zero count in `n_subsetCodelist`, `n_valueList`, or `n_assignedTerm` (beyond the TESTCD pin) is graph content the xlsx does not preserve.


In [12]:
rows = []
for dsid, d in all_dss.items():
    vs = d.get('variables', [])
    rows.append({
        'DS_Code':                dsid,
        'domain':                 d.get('domain'),
        'source':                 d.get('source'),
        'BC_ID':                  d.get('biomedicalConceptId'),
        'n_variables':            len(vs),
        'n_assignedTerm':         sum(1 for v in vs if v.get('assignedTerm')),
        'n_codelist':             sum(1 for v in vs if v.get('codelist')),
        'n_subsetCodelist':       sum(1 for v in vs if v.get('subsetCodelist')),
        'n_valueList':            sum(1 for v in vs if v.get('valueList')),
        'n_relationship':         sum(1 for v in vs if v.get('relationship')),
        'n_mandatoryValue_True':  sum(1 for v in vs if v.get('mandatoryValue') is True),
    })

inv = pd.DataFrame(rows).sort_values(['source','DS_Code']).reset_index(drop=True)
print(inv.shape)
inv.head(20)


(227, 11)


,DS_Code,domain,source,BC_ID,n_variables,n_assignedTerm,n_codelist,n_subsetCodelist,n_valueList,n_relationship,n_mandatoryValue_True
0,ADVEVENT,DS,DS.DECODE,C41331,5,2,3,0,1,5,2
1,DEAD,DS,DS.DECODE,C28554,5,2,3,0,1,5,2
2,DSSTUDYPARTICIPATION,DS,DS.DECODE,C224898,5,2,3,0,0,5,4
3,DSSTUDYTREATMENT,DS,DS.DECODE,C224899,5,2,3,1,1,5,4
4,FAILCONT,DS,DS.DECODE,C139236,5,2,3,0,0,5,2
5,FAILRAND,DS,DS.DECODE,C105448,5,2,3,0,0,5,2
6,LACKEFFICACY,DS,DS.DECODE,C48226,5,2,3,0,1,5,2
7,LTFUP,DS,DS.DECODE,C48227,5,2,3,0,1,5,2
8,PHYDECISION,DS,DS.DECODE,C48250,5,2,3,0,1,5,2
9,PROGDISEASE,DS,DS.DECODE,C35571,5,2,3,0,1,5,2


In [13]:
# Aggregate totals — at-a-glance reification across the package
print('package totals across all DSSs:')
for col in ('n_variables','n_assignedTerm','n_codelist','n_subsetCodelist',
            'n_valueList','n_relationship','n_mandatoryValue_True'):
    print(f'  {col:26s} {int(inv[col].sum()):6d}')

print()
print('DSSs with subsetCodelist on at least one variable (current xlsx drops this):')
print(f"  {(inv['n_subsetCodelist'] > 0).sum()} DSSs")

print()
print('DSSs with valueList on at least one variable (current xlsx drops this):')
print(f"  {(inv['n_valueList'] > 0).sum()} DSSs")

print()
print('DSSs with >1 assignedTerm (identity pin on variables beyond TESTCD):')
print(f"  {(inv['n_assignedTerm'] > 1).sum()} DSSs")


package totals across all DSSs:
  n_variables                  1761
  n_assignedTerm                619
  n_codelist                    854
  n_subsetCodelist                7
  n_valueList                   250
  n_relationship               1760
  n_mandatoryValue_True         458

DSSs with subsetCodelist on at least one variable (current xlsx drops this):
  7 DSSs

DSSs with valueList on at least one variable (current xlsx drops this):
  132 DSSs

DSSs with >1 assignedTerm (identity pin on variables beyond TESTCD):
  227 DSSs


## 8. One walked example per distinct `source` pattern

One DSS per `source` value, with its full variable shape. This is the qualitative view that complements the aggregate in §7 — does the flattener's assumed `VS.VSTESTCD`-style shape generalise, or are there source patterns it implicitly ignores?


In [14]:
for src in sorted(s for s in source_counts if s):
    example_id = next(k for k, d in all_dss.items() if d.get('source') == src)
    d = all_dss[example_id]
    print(f"=== source={src}  |  example DS_Code={example_id}  |  {d.get('shortName','')} ===")
    for v in d.get('variables', []):
        reified = [k for k in ('assignedTerm','codelist','subsetCodelist','valueList') if v.get(k)]
        at = v.get('assignedTerm') or {}
        print(f"  {v['name']:14s} role={v.get('role','-'):10s} "
              f"mVal={str(v.get('mandatoryValue','-')):5s} cmp={str(v.get('comparator','-')):4s} "
              f"pinned={str(at.get('value','')):12s} reified={','.join(reified)}")
    print()


=== source=DS.DECODE  |  example DS_Code=ADVEVENT  |  Adverse Event [RETIRED] ===
  DSCAT          role=Qualifier  mVal=False cmp=-    pinned=DISPOSITION EVENT reified=assignedTerm,codelist
  DSSCAT         role=Qualifier  mVal=False cmp=-    pinned=             reified=codelist,valueList
  DSDECOD        role=Qualifier  mVal=True  cmp=-    pinned=ADVERSE EVENT reified=assignedTerm,codelist
  DSTERM         role=Topic      mVal=True  cmp=-    pinned=             reified=
  DSSTDTC        role=Timing     mVal=False cmp=-    pinned=             reified=

=== source=GF.GFTESTCD  |  example DS_Code=CPNUMVARATIO  |  Copy Number Variation Ratio ===
  SPDEVID        role=Identifier mVal=False cmp=-    pinned=             reified=
  GFTESTCD       role=Topic      mVal=True  cmp=EQ   pinned=CPNUMVAR     reified=assignedTerm,codelist
  GFTEST         role=Qualifier  mVal=True  cmp=-    pinned=Copy Number Variation reified=assignedTerm,codelist
  GFTSTDTL       role=Qualifier  mVal=False cmp=EQ  

## 9. Validate all 227 DSSs against the schema

If this comes back clean, Step 2 can parse with plain `pyyaml` plus `linkml.validator.validate` for QC, no per-DSS exception handling needed.


In [15]:
failures = []
for dsid, d in all_dss.items():
    rep = validate(d, str(SDTM_SCHEMA), target_class='SDTMGroup')
    if rep.results:
        failures.append((dsid, rep.results))

print(f'validated {len(all_dss)} DSSs -> {len(failures)} with issues')
for dsid, results in failures[:20]:
    print(f'  {dsid}: {len(results)} issue(s)')
    for r in results[:3]:
        print(f'    {r.severity}: {r.message}')


validated 227 DSSs -> 0 with issues


## 10. What this tells us for Step 2

The cells below render a compact findings summary from the data computed above. Observations this notebook is designed to surface:

- How many distinct `source` patterns exist — drives whether the flattener can assume one shape or must generalise.
- How many DSSs use `subsetCodelist` or `valueList` — the XRAYCHEST-type patterns currently dropped by the xlsx.
- How many DSSs have `assignedTerm` on variables beyond TESTCD — identity pins on PRLOC, MKMETHOD etc. currently dropped.
- Whether all 227 DSSs validate against the schema as-published.

These numbers belong in the Step 2 design note, not in this notebook's narrative — the notebook is the evidence, the design note is the argument.


In [16]:
print('=== COSMoS 2026-Q1 SDTM DSS reconnaissance ===')
print(f'  package                         : yaml/{PACKAGE_DATE}/sdtm/')
print(f'  DSSs parsed                     : {len(all_dss)}')
print(f'  distinct source patterns        : {len(source_counts)}')
print(f'  DSSs with subsetCodelist>=1 var : {int((inv["n_subsetCodelist"] > 0).sum())}')
print(f'  DSSs with valueList>=1 var      : {int((inv["n_valueList"] > 0).sum())}')
print(f'  DSSs with assignedTerm>1 var    : {int((inv["n_assignedTerm"] > 1).sum())}')
print(f'  DSSs failing schema validation  : {len(failures)}')


=== COSMoS 2026-Q1 SDTM DSS reconnaissance ===
  package                         : yaml/20260331_r16/sdtm/
  DSSs parsed                     : 227
  distinct source patterns        : 5
  DSSs with subsetCodelist>=1 var : 7
  DSSs with valueList>=1 var      : 132
  DSSs with assignedTerm>1 var    : 227
  DSSs failing schema validation  : 0
